In [2]:
from src.core.preprocessing import encode_text
from typing import List
import numpy as np
from typing import Dict

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer

def build_tfidf(texts: List[str]):
    vectorizer = TfidfVectorizer(max_features=10000)
    X = vectorizer.fit_transform(texts)
    return X

In [4]:
def build_random_embeddings(texts: List[str], word2id: Dict[str,int], dim: int =100) -> np.array:
    random_matrix = np.random.randn(len(word2id), dim)

    X = []
    for text in texts:
        tokens = encode_text(text, word2id)
        vecs = random_matrix[tokens]
        X.append(vecs.mean(axis=0))

    return np.array(X)

In [5]:
def build_word2vec_embeddings(texts: List[str], embeddings: np.ndarray, word2id: Dict[str,int]):
    X = []
    for text in texts:
        tokens = encode_text(text, word2id)
        vecs = embeddings[tokens]
        X.append(vecs.mean(axis=0))

    return np.array(X)

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from scipy import sparse

def run_experiment(X: np.array, y: List[int], use_scaler: bool=True):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    if use_scaler:
        if sparse.issparse(X_train):
            scaler = StandardScaler(with_mean=False)
        else:
            scaler = StandardScaler()

        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)

    clf = LogisticRegression(max_iter=1000)
    clf.fit(X_train, y_train)

    pred = clf.predict(X_test)
    return accuracy_score(y_test, pred)

In [7]:
from datasets import load_dataset

dataset_news = load_dataset("sh0416/ag_news")
dataset_news = dataset_news.shuffle(seed=42)

In [8]:
from src.core.model_exp import *

model = Word2VecSGNS(
    vocab_size=0, dim=100, word2id={}, id2word={}, optimizer="adam"
)

model.load("../../../outputs/model/epoch_2_part_1.00_loss_2.2405.npz")

In [9]:
embeddings = model.V
word2id = model.word2id

In [10]:
texts = []
labels = []

for i in range(12000):
    example = dataset_news["train"][i]

    text = example["title"] + ". " + example["description"]
    label = int(example["label"]) - 1

    texts.append(text)
    labels.append(label)

In [11]:
def experiments(texts: List[str], labels: List[int], word2id: Dict[str,int], embeddings: np.ndarray):

    X_tfidf = build_tfidf(texts)
    acc_tfidf = run_experiment(X_tfidf, labels, use_scaler=False)

    # Random
    X_rand = build_random_embeddings(texts, word2id)
    acc_rand = run_experiment(X_rand, labels)

    # My embeddings
    X_word2vec = build_word2vec_embeddings(texts, embeddings, word2id)
    acc_word2vec = run_experiment(X_word2vec, labels)

    return acc_tfidf, acc_rand, acc_word2vec


In [12]:
acc_tfidf_ag, acc_rand_ag, acc_word2vec_ag = experiments(texts, labels, word2id, embeddings)

In [13]:
from datasets import load_dataset
dataset_review = load_dataset("acosio14/imbd-movie-reviews")
dataset_review.shuffle(seed=42)

DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'input_ids', 'attention_mask'],
        num_rows: 40000
    })
    test: Dataset({
        features: ['text', 'labels', 'input_ids', 'attention_mask'],
        num_rows: 10000
    })
})

In [14]:
texts = []
labels = []

for i in range(12000):
    example = dataset_review["train"][i]

    text = example["text"]
    label = int(example["labels"])

    texts.append(text)
    labels.append(label)

In [15]:
acc_tfidf_imdb, acc_rand_imdb, acc_word2vec_imdb = experiments(texts, labels, word2id, embeddings)

In [16]:
from datasets import load_dataset
dataset_wiki = load_dataset("fancyzhx/dbpedia_14")

In [17]:
dataset_wiki = dataset_wiki.shuffle(seed=42)

In [18]:
dataset_wiki["train"]

Dataset({
    features: ['label', 'title', 'content'],
    num_rows: 560000
})

In [19]:
texts = []
labels = []

for i in range(12000):
    example = dataset_wiki["train"][i]
    text = example["content"]
    label = int(example["label"])

    texts.append(text)
    labels.append(label)

In [20]:
acc_tfidf_wiki, acc_rand_wiki, acc_word2vec_wiki = experiments(texts, labels, word2id, embeddings)

In [28]:
import pandas as pd

results = pd.DataFrame({
    "Method": ["Random", "TF-IDF", "Word2Vec"],
    "AG News": [acc_rand_ag, acc_tfidf_ag, acc_word2vec_ag],
    "IMDB": [acc_rand_imdb, acc_tfidf_imdb, acc_word2vec_imdb],
    "Dbpedia": [acc_rand_wiki, acc_tfidf_wiki, acc_word2vec_wiki]
})

print(results)

     Method   AG News      IMDB   Dbpedia
0    Random  0.534583  0.714167  0.743750
1    TF-IDF  0.887917  0.883333  0.959583
2  Word2Vec  0.852083  0.748750  0.899167


In [35]:
results = results.round(2)

In [45]:
results.to_html("../../../outputs/docs/results.html", index=False)